# ManualChaidTree — Titanic demo

Build a CHAID survival tree on the classic **Titanic** dataset and test the core
(candidate ranking, split, score, save/load, export) end to end.

The dataset ships at the project root as `titanic.csv` (892 rows). If it is missing,
the first code cell downloads it from the public `datasciencedojo/datasets` mirror.

Target: **`Survived`** (1 = survived, 0 = died). Categorical features such as `Sex`
and `Embarked` are handled natively by the binning layer, and `Age` / `Embarked`
missing values fall into their own `IS NULL` bins.

In [1]:
import os, sys, urllib.request
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from sklearn.model_selection import train_test_split
from manual_chaid import ManualChaidTree

CSV = '../titanic.csv'
if not os.path.exists(CSV):
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    urllib.request.urlretrieve(url, CSV)

df = pd.read_csv(CSV)
# Drop identifiers / high-cardinality free text that don't bin meaningfully.
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
y = df.pop('Survived')
X = df

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)
X_train = X_train.reset_index(drop=True); y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True);  y_test  = y_test.reset_index(drop=True)
print(X_train.shape, 'survival rate =', round(y_train.mean(), 4))
X_train.head()

(668, 7) survival rate = 0.3832


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,female,35.0,1,0,90.000,S
1,2,male,19.0,0,0,10.500,S
2,2,male,34.0,0,0,13.000,S
3,3,female,4.0,0,2,22.025,S
4,2,female,21.0,0,0,10.500,S


## 1. Rank candidate root splits by chi-square

`min_samples` keeps bins from getting too small; `max_children` caps how many
branches a single split may produce. `candidate_splits` returns every column
scored and sorted (splittable first, then by χ² descending).

In [3]:
t = ManualChaidTree(X_train, y_train)
t.train(min_samples=40, max_children=4)

for c in t.candidate_splits(0):
    chi2 = c['chi2']
    chi2 = float('nan') if chi2 is None else chi2
    print(f"{c['variable']:<10} chi2={chi2:7.1f}  p={c['pvalue']:.2g}  "
          f"splittable={c['splittable']}  bins={len(c['conditions'])}")

Sex        chi2=  207.4  p=5.2e-47  splittable=True  bins=2
Fare       chi2=  102.2  p=5.1e-22  splittable=True  bins=4
Pclass     chi2=   84.7  p=4.1e-19  splittable=True  bins=3
Age        chi2=   31.7  p=2.2e-06  splittable=True  bins=5
Embarked   chi2=   28.7  p=2.6e-06  splittable=True  bins=4
Parch      chi2=   20.3  p=6.5e-06  splittable=True  bins=2
SibSp      chi2=    1.3  p=0.25  splittable=False  bins=2


## 2. Grow a small tree headlessly

Everything the UI does is available on the object. Here we greedily split each
leaf on its top significant candidate until a depth / size limit, then print the
tree. (Open the interactive builder instead with section 5.)

In [ ]:
MIN_SAMPLES, MAX_DEPTH = 40, 3

def grow(t, max_depth=MAX_DEPTH, min_samples=MIN_SAMPLES):
    changed = True
    while changed:
        changed = False
        for leaf in list(t.leaves()):
            if leaf.depth >= max_depth or leaf.n_samples < 2 * min_samples:
                continue
            best = next((c for c in t.candidate_splits(leaf.node_id)
                         if c['splittable']), None)
            if best:
                t.split(leaf.node_id, best['variable'], best['conditions'])
                changed = True

def show_tree(t):
    for nid in sorted(t.nodes):
        n = t.nodes[nid]
        rule = n.condition.describe() if n.condition else 'ROOT'
        extra = '' if n.is_leaf else (
            f"  | split={n.split_variable} chi2={n.split_chi2:.1f} p={n.split_pvalue:.2g}")
        print(f"{'  ' * n.depth}#{nid} [{rule}] n={n.n_samples} surv={n.target_rate:.1%}{extra}")

grow(t)
show_tree(t)
print(f"\ndepth={t.depth()}  leaves={len(t.leaves())}")

## 3. Evaluate: Gini on train vs. held-out test

In [4]:
print('train Gini:', round(t.gini(), 4))
print('test  Gini:', round(t.gini(X_test, y_test), 4))

preview = X_test.head().copy()
preview['leaf']  = t.predict_node(X_test.head())
preview['p_surv'] = t.predict_proba(X_test.head()).round(3)
preview

train Gini: 0.7386
test  Gini: 0.6348


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,leaf,p_surv
0,3,male,30.0,0,0,8.0500,S,14,0.130
1,3,female,21.0,0,0,7.7500,Q,16,0.725
2,3,male,15.0,1,1,7.2292,C,3,0.040
3,3,female,NaN,0,0,7.7875,Q,16,0.725
4,3,male,14.0,5,2,46.9000,S,5,0.313


## 4. Save / load, PNG, and rule export

`save` pickles structure + config only (no data). Reload **with** `X`/`y` to keep
editing, or **without** to get a view/score-only tree.

In [5]:
t.save('titanic_tree.pkl')

t_view = ManualChaidTree.load('titanic_tree.pkl')          # no data -> view/score only
print('scored new rows:', t_view.predict_proba(X_test.head()).round(3))

t_edit = ManualChaidTree.load('titanic_tree.pkl', X=X_train, y=y_train)
print('reloaded (with data) train Gini:', round(t_edit.gini(), 4))

rules = t.export_rules_excel('titanic_leaf_rules.xlsx')
rules

scored new rows: [0.13  0.725 0.04  0.725 0.313]
reloaded (with data) train Gini: 0.7386


,Node ID,Samples,Targets,Target Rate,Rule1,Rule2,Rule3
0,3,75,3,0.040000,Sex in [male],Fare < 7.744,
1,5,67,21,0.313433,Sex in [male],26.27 <= Fare < 52.28,
2,6,45,19,0.422222,Sex in [male],Fare >= 52.28,
3,8,54,49,0.907407,Sex in [female],1.5 <= Pclass < 2.5,
4,9,75,73,0.973333,Sex in [female],Pclass < 1.5,
5,10,64,5,0.078125,Sex in [male],7.744 <= Fare < 26.27,Age >= 32.5
6,13,45,14,0.311111,Sex in [male],7.744 <= Fare < 26.27,Age < 20.5
7,14,138,18,0.130435,Sex in [male],7.744 <= Fare < 26.27,Age IS NULL OR 20.5 <= Age < 32.5
8,15,65,25,0.384615,Sex in [female],Pclass >= 2.5,Fare >= 8.04
9,16,40,29,0.725000,Sex in [female],Pclass >= 2.5,Fare < 8.04


In [7]:
t_edit.train(min_samples=20, max_children=3)  # retrain with different parameters

In [8]:
t_view.train()

In [6]:
# Render the tree to a PNG (requires plotly + kaleido).
t.save_png('titanic_tree.png')

'titanic_tree.png'

## 5. (Optional) Launch the interactive builder

Open the tkinter window to build the tree by hand: pick a leaf, choose a
high-χ² variable, tweak cut-offs, **Split** / **Combine** / **Prune** / **Undo**,
then **Save** / **Export**. Close the window to return here. (Run on a local
machine with a display.)

In [ ]:
tree = ManualChaidTree(X_train, y_train)
tree.train(min_samples=40, max_children=4)  # launch_ui defaults to True